# Food Recognition Challenge 2026 — Progressive Resizing

Train a wider ResNet with progressive resizing (64 → 128 → 224) + mixup augmentation.

**Setup:** Kaggle Notebook → Settings → Accelerator → **GPU T4 x2**

In [ ]:
import os
import csv
import json
import time
import gc
import re
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Config

**Important:** Update `DATA_DIR` below to match your Kaggle dataset path.

When you add the competition data as input in Kaggle, it will typically be at `/kaggle/input/food-recognition-challenge-2026/`.

In [ ]:
# ── Paths — UPDATE THESE if your Kaggle dataset name differs ──
DATA_DIR = Path('/kaggle/input/food-recognition-challenge-2026')
TRAIN_IMG_DIR = DATA_DIR / 'train_set' / 'train_set' / 'train_set'
TEST_IMG_DIR = DATA_DIR / 'test_set' / 'test_set' / 'test_set'
TRAIN_LABELS_CSV = DATA_DIR / 'train_labels.csv'
CLASS_LIST = DATA_DIR / 'class_list_food.txt'
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ──
NUM_CLASSES = 80
DATASET_MEAN = [0.6310, 0.5407, 0.4424]
DATASET_STD = [0.2259, 0.2413, 0.2647]

# ── Training ──
VAL_SPLIT = 0.2
SEED = 42
NUM_WORKERS = 2  # Kaggle supports multiprocessing

# ── Device ──
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Verify paths exist
for p, name in [(TRAIN_IMG_DIR, 'Train images'), (TEST_IMG_DIR, 'Test images'), 
                (TRAIN_LABELS_CSV, 'Labels CSV'), (CLASS_LIST, 'Class list')]:
    print(f'{name}: {"OK" if p.exists() else "NOT FOUND — check DATA_DIR!"} ({p})')

## Dataset & Augmentation

In [ ]:
def get_transforms(mode='train', img_size=128, augmentation='basic'):
    normalize = transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD)

    if mode in ('val', 'test'):
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            normalize,
        ])

    aug_list = [transforms.Resize((img_size + 16, img_size + 16))]

    if augmentation in ('basic', 'medium', 'heavy'):
        aug_list += [transforms.RandomCrop(img_size), transforms.RandomHorizontalFlip()]
    else:
        aug_list += [transforms.CenterCrop(img_size)]

    if augmentation in ('medium', 'heavy'):
        aug_list += [
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.RandomRotation(15),
        ]

    if augmentation == 'heavy':
        aug_list += [
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.RandomGrayscale(p=0.05),
        ]

    aug_list += [transforms.ToTensor(), normalize]

    if augmentation == 'heavy':
        aug_list.append(transforms.RandomErasing(p=0.2))

    return transforms.Compose(aug_list)


class FoodDataset(Dataset):
    def __init__(self, img_paths, labels=None, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, self.labels[idx] - 1  # 1-80 → 0-79
        return img


def load_train_data():
    img_paths, labels = [], []
    with open(TRAIN_LABELS_CSV) as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_paths.append(TRAIN_IMG_DIR / row['img_name'])
            labels.append(int(row['label']))
    return img_paths, labels


def load_test_data():
    test_files = sorted(
        TEST_IMG_DIR.iterdir(),
        key=lambda p: int(re.search(r'(\d+)', p.stem).group(1)),
    )
    return test_files, [p.name for p in test_files]


def get_class_weights(labels):
    counts = np.bincount(np.array(labels) - 1, minlength=NUM_CLASSES).astype(np.float32)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * NUM_CLASSES
    return torch.FloatTensor(weights)


def get_dataloaders(img_size=128, batch_size=64, augmentation='basic'):
    img_paths, labels = load_train_data()
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        img_paths, labels, test_size=VAL_SPLIT, random_state=SEED, stratify=labels,
    )

    train_ds = FoodDataset(train_paths, train_labels, get_transforms('train', img_size, augmentation))
    val_ds = FoodDataset(val_paths, val_labels, get_transforms('val', img_size))

    # Weighted sampler for class imbalance
    class_weights = get_class_weights(train_labels)
    sample_weights = class_weights[torch.LongTensor(train_labels) - 1]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader


# Quick test
img_paths, labels = load_train_data()
print(f'Train images: {len(img_paths)}, Classes: {len(set(labels))}')

## Model: FoodResNetLarge

Wider ResNet (64→128→256→512 channels) with residual blocks.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out, inplace=True)


class FoodResNetLarge(nn.Module):
    """Wider ResNet: 64 -> 128 -> 256 -> 512 channels."""

    def __init__(self, num_classes=NUM_CLASSES, dropout=0.5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self._init_weights()

    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def get_features(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)


model = FoodResNetLarge()
params = sum(p.numel() for p in model.parameters())
print(f'FoodResNetLarge parameters: {params:,}')
del model

## Trainer with Mixup

In [ ]:
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


class Trainer:
    def __init__(self, model, train_loader, val_loader, experiment_name='exp',
                 epochs=25, lr=1e-3, scheduler_type='onecycle',
                 label_smoothing=0.0, mixup_alpha=0.0):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.experiment_name = experiment_name
        self.epochs = epochs
        self.lr = lr
        self.mixup_alpha = mixup_alpha

        # Loss with class weights
        all_labels = train_loader.dataset.labels
        weights = get_class_weights(all_labels).to(DEVICE)
        self.criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)

        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        if scheduler_type == 'onecycle':
            self.scheduler = OneCycleLR(
                self.optimizer, max_lr=lr,
                steps_per_epoch=len(train_loader), epochs=epochs,
            )
            self.step_per_batch = True
        elif scheduler_type == 'cosine':
            self.scheduler = CosineAnnealingLR(self.optimizer, T_max=epochs, eta_min=1e-6)
            self.step_per_batch = False
        else:
            self.scheduler = None
            self.step_per_batch = False

        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
        self.best_val_acc = 0.0
        self.exp_dir = OUTPUT_DIR / experiment_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)

    def train_epoch(self):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for imgs, labels in self.train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            self.optimizer.zero_grad()

            if self.mixup_alpha > 0:
                mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, self.mixup_alpha)
                outputs = self.model(mixed_imgs)
                loss = mixup_criterion(self.criterion, outputs, y_a, y_b, lam)
            else:
                outputs = self.model(imgs)
                loss = self.criterion(outputs, labels)

            loss.backward()
            self.optimizer.step()

            if self.step_per_batch and self.scheduler:
                self.scheduler.step()

            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
        return total_loss / total, correct / total

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []
        for imgs, labels in self.val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = self.model(imgs)
            loss = self.criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

    def train(self):
        print(f'\n{"="*60}')
        print(f'Experiment: {self.experiment_name}')
        print(f'Parameters: {sum(p.numel() for p in self.model.parameters()):,}')
        print(f'Epochs: {self.epochs}, LR: {self.lr}, Mixup: {self.mixup_alpha}')
        print(f'{"="*60}\n')

        start = time.time()
        for epoch in range(1, self.epochs + 1):
            t0 = time.time()
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc, _, _ = self.validate()

            if self.scheduler and not self.step_per_batch:
                self.scheduler.step()

            lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['lr'].append(lr)

            elapsed = time.time() - t0
            print(f'Epoch {epoch:3d}/{self.epochs} | '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
                  f'LR: {lr:.6f} | {elapsed:.1f}s')

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                torch.save(self.model.state_dict(), self.exp_dir / 'best_model.pt')
                print(f'  -> New best val accuracy: {val_acc:.4f}')

        total = time.time() - start
        print(f'\nTraining complete in {total/60:.1f} min. Best val acc: {self.best_val_acc:.4f}')

        with open(self.exp_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)

        return self.history

## Progressive Resizing: 64 → 128 → 224

In [ ]:
stages = [
    {'img_size': 64,  'epochs': 25, 'lr': 2e-3, 'batch_size': 128, 'augmentation': 'basic',  'mixup': 0.0},
    {'img_size': 128, 'epochs': 25, 'lr': 1e-3, 'batch_size': 64,  'augmentation': 'medium', 'mixup': 0.2},
    {'img_size': 224, 'epochs': 30, 'lr': 5e-4, 'batch_size': 32,  'augmentation': 'medium', 'mixup': 0.4},
]

torch.manual_seed(SEED)
np.random.seed(SEED)

model = FoodResNetLarge()

for i, stage in enumerate(stages):
    stage_name = f'stage{i+1}_{stage["img_size"]}px'
    print(f'\n{"#"*60}')
    print(f'# Stage {i+1}/3: {stage["img_size"]}x{stage["img_size"]}')
    print(f'# Epochs: {stage["epochs"]}, LR: {stage["lr"]}, Batch: {stage["batch_size"]}')
    print(f'# Augmentation: {stage["augmentation"]}, Mixup: {stage["mixup"]}')
    print(f'{"#"*60}')

    train_loader, val_loader = get_dataloaders(
        img_size=stage['img_size'],
        batch_size=stage['batch_size'],
        augmentation=stage['augmentation'],
    )

    trainer = Trainer(
        model, train_loader, val_loader,
        experiment_name=stage_name,
        epochs=stage['epochs'],
        lr=stage['lr'],
        scheduler_type='onecycle',
        label_smoothing=0.1,
        mixup_alpha=stage['mixup'],
    )
    history = trainer.train()

    # Load best weights for next stage
    best_path = trainer.exp_dir / 'best_model.pt'
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f'\nLoaded best model from {stage_name} (val acc: {trainer.best_val_acc:.4f})')

    # Cleanup
    del trainer, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('Progressive resizing complete!')
print('='*60)

## Evaluation & Visualizations

In [ ]:
# Load best model from final stage
best_path = OUTPUT_DIR / 'stage3_224px' / 'best_model.pt'
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Evaluate on validation set at 224px
_, val_loader = get_dataloaders(img_size=224, batch_size=32, augmentation='basic')

correct, total = 0, 0
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_acc = correct / total
print(f'Final validation accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)')

In [ ]:
# Plot training curves for all stages
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2196F3', '#FF9800', '#4CAF50']
offset = 0

for i, stage in enumerate(stages):
    stage_name = f'stage{i+1}_{stage["img_size"]}px'
    hist_path = OUTPUT_DIR / stage_name / 'history.json'
    if not hist_path.exists():
        continue
    with open(hist_path) as f:
        h = json.load(f)

    epochs = range(offset + 1, offset + len(h['train_loss']) + 1)
    label = f'{stage["img_size"]}px'

    axes[0].plot(epochs, h['train_loss'], color=colors[i], label=f'Train {label}', linestyle='-')
    axes[0].plot(epochs, h['val_loss'], color=colors[i], label=f'Val {label}', linestyle='--')

    axes[1].plot(epochs, h['train_acc'], color=colors[i], label=f'Train {label}', linestyle='-')
    axes[1].plot(epochs, h['val_acc'], color=colors[i], label=f'Val {label}', linestyle='--')

    axes[2].plot(epochs, h['lr'], color=colors[i], label=label)

    offset += len(h['train_loss'])

axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Progressive Resizing: 64px -> 128px -> 224px', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'progressive_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Generate Submission

In [ ]:
# Generate test predictions
test_paths, test_names = load_test_data()
test_tf = get_transforms('test', img_size=224)
test_ds = FoodDataset(test_paths, transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

all_preds = []
model.eval()
with torch.no_grad():
    for imgs in test_loader:
        if isinstance(imgs, (list, tuple)):
            imgs = imgs[0]
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1)
        all_preds.extend((preds + 1).cpu().numpy())  # back to 1-80

# Save submission
submission_path = OUTPUT_DIR / 'submission_progressive.csv'
with open(submission_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['img_name', 'label'])
    for name, pred in zip(test_names, all_preds):
        writer.writerow([name, pred])

print(f'Submission saved: {submission_path}')
print(f'Total test predictions: {len(all_preds)}')

# Also save the model
torch.save(model.state_dict(), OUTPUT_DIR / 'final_progressive_model.pt')
print('Model saved!')

In [ ]:
# Download files from Kaggle
from IPython.display import FileLink
display(FileLink(str(submission_path)))